[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hentablihamza/fingerprint-similarity-guide/blob/main/notebooks/03_similarity_metrics.ipynb)

# 03 · Similarity Metrics Explained

*Fingerprint & Similarity Guide · Hamza Hentabli, Ph.D. · ~40 min*

A fingerprint turns a molecule into a set of on-bits. A **similarity metric** turns two such sets into one number. Tanimoto is the reflex choice, but Dice, Cosine and the asymmetric Tversky coefficient answer subtly different questions — and Euclidean distance quietly gives the wrong answer for sparse bits. We work the math on tiny examples so the formulas stop being mysterious.

**Learning objectives.** By the end of this notebook you will be able to:

- write down and compute Tanimoto, Dice and Cosine from bit overlaps by hand
- use the **asymmetric Tversky** coefficient for substructure-style search
- explain why **Euclidean distance** is the wrong tool for sparse fingerprints
- match a metric to the question you are actually asking

In [1]:
# Setup: on Google Colab this installs dependencies and clones the repo so the
# cached data and the shared `fsg` helper module are available. Running locally,
# install requirements.txt once and this cell just locates the files.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
OFFLINE = os.environ.get("FSG_OFFLINE", "0") == "1"   # CI sets this; no notebook needs the network
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "rdkit", "scikit-learn", "scipy", "pandas", "matplotlib"], check=True)
    if not os.path.isdir("fingerprint-similarity-guide"):
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/hentablihamza/fingerprint-similarity-guide.git"], check=True)
    ROOT = "fingerprint-similarity-guide"
else:
    ROOT = ".." if os.path.isdir("../data") else "."
DATA_DIR = os.path.join(ROOT, "data")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import fsg   # shared fingerprint factory + metrics + evaluation

print("Colab:", IN_COLAB, "| offline:", OFFLINE, "| data:", DATA_DIR)
print("Fingerprints available:", ", ".join(fsg.FP_NAMES))

Colab: False | offline: True | data: ../data
Fingerprints available: ECFP4-1024, ECFP4-2048, ECFP6-2048, ECFP4-counts, FCFP4-2048, RDKit, AtomPair, Torsion, MACCS, Avalon


## 1. The building blocks: a, b, c

For two binary fingerprints, let **a** = bits on in A only, **b** = bits on in B only, **c** = bits on in *both*. Then:

- **Tanimoto** (Jaccard) = c / (a + b + c) — shared / total
- **Dice** = 2c / (2c + a + b) — shared, counted double
- **Cosine** = c / √((a+c)(b+c))

Let's verify RDKit matches these formulas on two small Morgan fingerprints.

In [2]:
from rdkit import Chem, DataStructs
m1 = Chem.MolFromSmiles("CCO")       # ethanol
m2 = Chem.MolFromSmiles("CCOC")      # methoxyethane
fp1, fp2 = fsg.make_fp("ECFP4-2048", m1), fsg.make_fp("ECFP4-2048", m2)

on1 = set(fp1.GetOnBits()); on2 = set(fp2.GetOnBits())
c = len(on1 & on2); a = len(on1 - on2); b = len(on2 - on1)
tani = c / (a + b + c)
dice = 2*c / (2*c + a + b)
import math
cos = c / math.sqrt((a+c) * (b+c))
print(f"by hand : a={a} b={b} c={c} | Tanimoto={tani:.3f} Dice={dice:.3f} Cosine={cos:.3f}")
print(f"RDKit   : Tanimoto={DataStructs.TanimotoSimilarity(fp1,fp2):.3f} "
      f"Dice={DataStructs.DiceSimilarity(fp1,fp2):.3f} Cosine={DataStructs.CosineSimilarity(fp1,fp2):.3f}")

by hand : a=3 b=5 c=3 | Tanimoto=0.273 Dice=0.429 Cosine=0.433
RDKit   : Tanimoto=0.273 Dice=0.429 Cosine=0.433


The hand math and RDKit agree exactly. Note **Dice ≥ Tanimoto** always (it weights shared bits more), while Cosine sits in between.

## 2. Tanimoto vs Dice vs Cosine across a real range

They rank pairs *almost* identically but compress the scale differently. We compare them on EGFR drug pairs.

In [3]:
import pandas as pd
drugs = {"gefitinib":"COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1",
         "erlotinib":"C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1",
         "lapatinib":"CS(=O)(=O)CCNCc1oc(-c2ccc3ncnc(Nc4ccc(OCc5cccc(F)c5)c(Cl)c4)c3c2)cc1",
         "osimertinib":"C=CC(=O)Nc1cc(Nc2nccc(-c3cn(C)c4ccccc34)n2)c(OC)cc1N(C)CCN(C)C"}
mols = {k: Chem.MolFromSmiles(v) for k, v in drugs.items()}
fps = {k: fsg.make_fp("ECFP4-2048", m) for k, m in mols.items()}
pairs = [("gefitinib","erlotinib"), ("gefitinib","lapatinib"), ("erlotinib","osimertinib")]
rows = [(x, y,
         round(fsg.similarity("tanimoto", fps[x], fps[y]), 3),
         round(fsg.similarity("dice", fps[x], fps[y]), 3),
         round(fsg.similarity("cosine", fps[x], fps[y]), 3)) for x, y in pairs]
pd.DataFrame(rows, columns=["A", "B", "Tanimoto", "Dice", "Cosine"])

,A,B,Tanimoto,Dice,Cosine
0,gefitinib,erlotinib,0.408,0.579,0.584
1,gefitinib,lapatinib,0.317,0.481,0.484
2,erlotinib,osimertinib,0.158,0.274,0.279


## 3. Tversky: asymmetric, for substructure search

Tversky generalises Tanimoto with two weights: **α** on the query's unique bits and **β** on the target's. With **α=1, β=0** the score asks *"is the query contained in the target?"* — ideal for substructure / scaffold queries, where a small query should score high against a large molecule that contains it. Tanimoto would unfairly penalise the size difference.

In [4]:
core = Chem.MolFromSmiles("c1ccc2ncncc2c1")             # quinazoline-like core (small)
big  = Chem.MolFromSmiles("COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1")  # gefitinib (large)
fcore, fbig = fsg.make_fp("ECFP4-2048", core), fsg.make_fp("ECFP4-2048", big)
print("Tanimoto (symmetric)          :", round(fsg.similarity("tanimoto", fcore, fbig), 3))
print("Tversky a=1,b=0 (query in tgt):", round(fsg.similarity("tversky", fcore, fbig, alpha=1.0, beta=0.0), 3))
print("Tversky a=0,b=1 (tgt in query):", round(fsg.similarity("tversky", fcore, fbig, alpha=0.0, beta=1.0), 3))

Tanimoto (symmetric)          : 0.111
Tversky a=1,b=0 (query in tgt): 0.4
Tversky a=0,b=1 (tgt in query): 0.133


With α=1, β=0 the small core scores highly against the large molecule that contains it — the asymmetry captures "contained in", which symmetric Tanimoto cannot.

## 4. Why NOT Euclidean distance on sparse bits

Euclidean distance counts *disagreements* (a + b). For sparse fingerprints almost every bit is 0 in both molecules, so two totally unrelated molecules look "close" simply because they share thousands of **off**-bits. Similarity metrics above deliberately ignore the 0–0 matches. Let's see the trap.

In [5]:
import numpy as np
from rdkit import DataStructs
def to_np(fp):
    arr = np.zeros((fp.GetNumBits(),), dtype=int); DataStructs.ConvertToNumpyArray(fp, arr); return arr
unrelated1 = fsg.make_fp("ECFP4-2048", Chem.MolFromSmiles("CCCCCCCC"))        # octane
unrelated2 = fsg.make_fp("ECFP4-2048", Chem.MolFromSmiles("c1ccc2ccccc2c1"))  # naphthalene
v1, v2 = to_np(unrelated1), to_np(unrelated2)
euclid = np.sqrt(((v1 - v2) ** 2).sum())
print("shared 0-bits:", int(((v1 == 0) & (v2 == 0)).sum()), "of 2048")
print("Euclidean distance:", round(float(euclid), 2), "-> looks 'small' only because of shared zeros")
print("Tanimoto (ignores 0-0):", round(fsg.similarity("tanimoto", unrelated1, unrelated2), 3))

shared 0-bits: 2032 of 2048
Euclidean distance: 4.0 -> looks 'small' only because of shared zeros
Tanimoto (ignores 0-0): 0.0


Thousands of shared **off**-bits make the Euclidean distance misleadingly small. Tanimoto, which only counts shared **on**-bits relative to the union, correctly reports these two molecules as dissimilar. **Rule: use set-overlap metrics, not Euclidean, on bit fingerprints.**

## 5. Russel-Rao and the rest

`fsg` also exposes **Russel-Rao** (shared on-bits / *total* bits), which behaves very differently and we will see win or lose depending on the task in the benchmarks. The lesson is not "metric X is best" but "metric choice is a real, measurable decision."

## Summary

- Tanimoto, Dice and Cosine are all built from shared/unique **on**-bits; Dice ≥ Tanimoto.
- **Tversky (α=1, β=0)** is the right tool for *substructure/scaffold* queries.
- **Euclidean distance is wrong** for sparse fingerprints — it rewards shared zeros.
- Metric choice is a genuine decision, which the next four benchmark notebooks quantify.


### ✏️ Exercise 1

Confirm Dice ≥ Tanimoto for all three EGFR drug pairs above.

<details>
<summary><b>Show solution</b></summary>

```python
for x, y in pairs:
    t = fsg.similarity("tanimoto", fps[x], fps[y]); d = fsg.similarity("dice", fps[x], fps[y])
    print(x, y, round(d - t, 3), ">=0:", d >= t - 1e-9)
```

</details>

### ✏️ Exercise 2

For the quinazoline core vs gefitinib, which α,β makes Tversky closest to 1.0, and what does that tell you about containment direction?

<details>
<summary><b>Show solution</b></summary>

```python
for a, b in [(1,0),(0.9,0.1),(0.5,0.5),(0,1)]:
    print(a, b, round(fsg.similarity("tversky", fcore, fbig, alpha=a, beta=b), 3))
# alpha high, beta low -> "query contained in target" scores highest.
```

</details>